## Setup

In [1]:
## Imports

from latincyreaders import TesseraeReader, AnnotationLevel

from pprint import pprint

In [2]:
## Set up reader

T = TesseraeReader()

## File Discovery

In [3]:
## First 8 filenames

files = T.fileids()[:8]
pprint(files)

['texts/ammianus.rerum_gestarum.part.14.tess',
 'texts/ammianus.rerum_gestarum.part.15.tess',
 'texts/ammianus.rerum_gestarum.part.16.tess',
 'texts/ammianus.rerum_gestarum.part.17.tess',
 'texts/ammianus.rerum_gestarum.part.18.tess',
 'texts/ammianus.rerum_gestarum.part.19.tess',
 'texts/ammianus.rerum_gestarum.part.20.tess',
 'texts/ammianus.rerum_gestarum.part.21.tess']


In [4]:
# Get all files
all_files = T.fileids()
print(f"Total files: {len(all_files)}")

Total files: 826


## Metadata

In [5]:
# Metadata can be accessed without NLP processing (instant)
# Use get_metadata() to avoid loading the spaCy model

catullus = T.fileids(match='catullus.carmina')[0]

# Fast: get metadata directly (no NLP overhead)
meta = T.get_metadata(catullus)

print(f"Metadata for {catullus}:")
pprint(meta)

Metadata for texts/catullus.carmina.tess:
{'author': 'Catullus',
 'author_identifier': 'http://www.wikidata.org/entity/Q163079',
 'date': '-54',
 'genre': 'lyric',
 'mode': 'verse',
 'url': 'https://github.com/latincy/lat_text_tesserae/blob/main/texts/catullus.carmina.tess',
 'wd_author_label': 'Catullus',
 'wd_author_pob': 'Verona',
 'wd_author_pob_id': 'http://www.wikidata.org/entity/Q2028',
 'wd_author_yob': '-83',
 'wd_author_yod': '-53'}


## Core Interface

### texts()

In [6]:
catullus = T.fileids(match='catullus.carmina')[0]

catullus_text = next(T.texts(catullus))
print(catullus_text[:500])

Cui dono lepidum novum libellum arido modo pumice expolitum? Corneli, tibi; namque tu solebas meas esse aliquid putare nugas, iam tum cum ausus es unus Italorum omne aevum tribus explicare chartis, doctis, Iuppiter, et laboriosis! quare habe tibi quidquid hoc libelli qualecumque, quod, o patrona virgo, plus uno maneat perenne saeclo. Passer, deliciae meae puellae, quicum ludere, quem in sinu tenere, cui primum digitum dare adpetenti et acris solet incitare morsus, cum desiderio meo nitenti carum


### docs()

In [7]:
catullus_doc = next(T.docs(catullus))
print(catullus_doc.text[:200])

Cui dono lepidum nouum libellum arido modo pumice expolitum? Corneli, tibi; namque tu solebas meas esse aliquid putare nugas, iam tum cum ausus es unus Italorum omne aeuum tribus explicare chartis, do


In [8]:
# doc metadata
print(catullus_doc._.metadata)

{'date': '-54', 'mode': 'verse', 'genre': 'lyric', 'author': 'Catullus', 'author_identifier': 'http://www.wikidata.org/entity/Q163079', 'url': 'https://github.com/latincy/lat_text_tesserae/blob/main/texts/catullus.carmina.tess', 'wd_author_label': 'Catullus', 'wd_author_yob': '-83', 'wd_author_yod': '-53', 'wd_author_pob_id': 'http://www.wikidata.org/entity/Q2028', 'wd_author_pob': 'Verona', 'filename': 'catullus.carmina.tess', 'path': '/Users/pjb311/latincy_data/lat_text_tesserae/texts/catullus.carmina.tess'}


### sents()

In [9]:
# Another file for examples
catilinam = T.fileids(match='cicero.in_catilinam')[0]

In [10]:
# Sents - spaCy Span objects (la_core_web_lg)
catilinam_sents = T.sents(catilinam)
sent = next(catilinam_sents)
print(sent)

quo usque tandem abutere, Catilina, patientia nostra?


In [11]:
# Line citations (sents can span multiple citation lines)
aeneid = T.fileids(match='aeneid')[0]
aeneid_lines = T.lines(aeneid)
for i in range(1, 6):
    line = next(aeneid_lines)
    print(f'{line._.citation}: {line}')

<verg. aen. 1.1>: Arma uirumque cano, Troiae qui primus ab oris
<verg. aen. 1.2>: Italiam, fato profugus, Lauiniaque uenit
<verg. aen. 1.3>: litora, multum ille et terris iactatus et alto
<verg. aen. 1.4>: ui superum saeuae memorem Iunonis ob iram;
<verg. aen. 1.5>: multa quoque et bello passus, dum conderet urbem,


### tokens()

In [12]:
# Tokens - list of first 8 spaCy Token objects
catilinam_tokens = list(T.tokens(catilinam))[:8]
print(catilinam_tokens)

[quo, usque, tandem, abutere, ,, Catilina, ,, patientia]


In [13]:
# Token attributes
tok = catilinam_tokens[0]
print(f"text: {tok.text}, lemma: {tok.lemma_}, pos: {tok.pos_}, tag: {tok.tag_}")

text: quo, lemma: quo, pos: ADV, tag: adverb


## LatinCy Annotations

Every sentence from `sents()` is a spaCy `Span`, so the full LatinCy annotation stack — lemmata, part-of-speech tags, morphological features, dependency relations, and named entities — is available directly on each token, with no extra processing step. Below we read Seneca, *Epistulae Morales* 1 token by token, dashboard-style, then turn to a Ciceronian speech to see named-entity recognition on a text that actually supplies names.

In [14]:
# Seneca, EM 1 — select sentences by citation from a single sents() call
seneca = T.fileids(match='epistulae_morales.part.1.1-10')[0]

def citation(sent):
    """Citation of the line where a sentence begins."""
    for line in sent.doc.spans['lines']:
        if line.start <= sent.start < line.end:
            return line._.citation
    return ''

em1_sents = [sent for sent in T.sents(seneca) if citation(sent).startswith('<sen.ep. 1.')]

print(f'Seneca, EM 1: {len(em1_sents)} sentences\n')
for sent in em1_sents[:4]:
    print(f'{citation(sent)}: {sent.text[:70]}...')

Seneca, EM 1: 29 sentences

<sen.ep. 1.0>: Seneca Lucilio suo salutem Ita fac, mi Lucili;...
<sen.ep. 1.1>: uindica te tibi, et tempus, quod adhuc aut auferebatur aut subripiebat...
<sen.ep. 1.1>: Persuade tibi hoc sic esse, ut scribo:...
<sen.ep. 1.1>: quaedam tempora eripiuntur nobis, quaedam subducuntur, quaedam effluun...


In [15]:
# Dashboard-style view: lemma, POS, dependency syntax, and morphology per token
sent = em1_sents[1]  # 'uindica te tibi...' — the letter's opening imperative
print(f'{citation(sent)}: {sent.text}\n')

print(f"{'form':<15}{'lemma':<12}{'upos':<7}{'dep':<12}{'head':<13}feats")
print('-' * 100)
for token in sent:
    print(f'{token.text:<15}{token.lemma_:<12}{token.pos_:<7}'
          f'{token.dep_:<12}{token.head.text:<13}{token.morph}')

<sen.ep. 1.1>: uindica te tibi, et tempus, quod adhuc aut auferebatur aut subripiebatur aut excidebat, collige et serua.

form           lemma       upos   dep         head         feats
----------------------------------------------------------------------------------------------------
uindica        uindico     VERB   ROOT        uindica      Aspect=Imp|Mood=Imp|Number=Sing|Person=2|Tense=Pres|VerbForm=Fin|Voice=Act
te             tu          PRON   obj         uindica      Case=Acc|Number=Sing|Person=2
tibi           tu          PRON   obl         uindica      Case=Dat|Number=Sing|Person=2
,              ,           PUNCT  punct       tempus       
et             et          CCONJ  cc          tempus       
tempus         tempus      NOUN   conj        tibi         Case=Acc|Gender=Neut|Number=Sing
,              ,           PUNCT  punct       auferebatur  
quod           qui         PRON   nsubj:pass  auferebatur  Case=Nom|Gender=Neut|Number=Sing
adhuc          adhuc       ADV    ad

In [16]:
# Aggregate over annotations: most frequent content lemmas in the letter
# ('tempus' tops the list — fitting for a letter on reclaiming one's time)
from collections import Counter

lemma_counts = Counter(
    token.lemma_
    for sent in em1_sents
    for token in sent
    if token.pos_ in {'NOUN', 'VERB', 'ADJ'}
)
lemma_counts.most_common(10)

[('tempus', 6),
 ('facio', 4),
 ('magnus', 3),
 ('uita', 3),
 ('ago', 3),
 ('seruo', 2),
 ('scribo', 2),
 ('fio', 2),
 ('uolo', 2),
 ('pars', 2)]

### Named entities

A philosophical letter names almost no one, so NER has little to find in Seneca. Point the *same* `sents()` interface at a political speech — Cicero's *First Catilinarian* — and the entities come pouring out, each still typed and still carrying its citation. The `citation()` helper above works unchanged, since it reads any span's `doc.spans['lines']`.

In [17]:
# The same annotation access, on a text full of names: Cicero, In Catilinam
catilina = T.fileids(match='cicero.in_catilinam')[0]
catil_sents = list(T.sents(catilina))

print('Named entities in the exordium of the First Catilinarian:\n')
shown = 0
for sent in catil_sents:
    cit = citation(sent)
    if not cit.startswith('<cic. catil. 1.'):
        continue
    for ent in sent.ents:
        print(f'{cit}: {ent.text:<13} ({ent.label_})')
        shown += 1
    if shown >= 16:
        break

# NER is a per-span annotation like any other — aggregate it across the whole speech
# Group by lemma to collapse inflectional variants (Catilina, Catilinam, Catilinae → lemma Catilina)
people_by_lemma = {}
for sent in catil_sents:
    for ent in sent.ents:
        if ent.label_ == 'PERSON':
            lemma = ent[0].lemma_  # lemma of first token
            people_by_lemma[lemma] = people_by_lemma.get(lemma, 0) + 1

labels = Counter(ent.label_ for sent in catil_sents for ent in sent.ents)
print(f'\nAcross all four orations: {dict(labels)}')
print(f'Most-named people (by lemma): {sorted(people_by_lemma.items(), key=lambda x: -x[1])[:5]}')

Named entities in the exordium of the First Catilinarian:

<cic. catil. 1.1>: Catilina      (PERSON)
<cic. catil. 1.1>: Palati        (LOC)
<cic. catil. 1.2>: Catilina      (PERSON)
<cic. catil. 1.3>: P. Scipio     (PERSON)
<cic. catil. 1.3>: Ti. Gracchum  (PERSON)
<cic. catil. 1.3>: Catilinam     (PERSON)
<cic. catil. 1.3>: C. Seruilius  (PERSON)
<cic. catil. 1.3>: Sp. Maelium   (PERSON)
<cic. catil. 1.3>: Catilina      (PERSON)
<cic. catil. 1.4>: L. Opimius    (PERSON)
<cic. catil. 1.4>: C. Gracchus   (PERSON)
<cic. catil. 1.4>: M. Fuluius    (PERSON)
<cic. catil. 1.4>: C. Mario      (PERSON)
<cic. catil. 1.4>: L. Ualerio    (PERSON)
<cic. catil. 1.4>: L. Saturninum (PERSON)
<cic. catil. 1.4>: C. Seruilium  (PERSON)



Across all four orations: {'PERSON': 231, 'LOC': 55, 'NORP': 83}
Most-named people (by lemma): [('Catilina', 62), ('Lucius', 19), ('Gaius', 18), ('Publius', 16), ('Lentulus', 12)]


## Tesserae Features

### lines()

In [18]:
# Lines (citation units from the Tesserae format)

aeneid = T.fileids(match='aeneid')[0]

aeneid_lines = T.lines(aeneid)

for i in range(1, 9):
    print(f'{i}: {next(aeneid_lines)}')

1: Arma uirumque cano, Troiae qui primus ab oris
2: Italiam, fato profugus, Lauiniaque uenit
3: litora, multum ille et terris iactatus et alto
4: ui superum saeuae memorem Iunonis ob iram;
5: multa quoque et bello passus, dum conderet urbem,
6: inferretque deos Latio, genus unde Latinum,
7: Albanique patres, atque altae moenia Romae.
8: Musa, mihi causas memora, quo numine laeso,


In [19]:
# Lines with citation information preserved

aeneid_lines = T.lines(aeneid)

for i in range(1, 9):
    line = next(aeneid_lines)
    print(f'{line._.citation}: {line}')

<verg. aen. 1.1>: Arma uirumque cano, Troiae qui primus ab oris
<verg. aen. 1.2>: Italiam, fato profugus, Lauiniaque uenit
<verg. aen. 1.3>: litora, multum ille et terris iactatus et alto
<verg. aen. 1.4>: ui superum saeuae memorem Iunonis ob iram;
<verg. aen. 1.5>: multa quoque et bello passus, dum conderet urbem,
<verg. aen. 1.6>: inferretque deos Latio, genus unde Latinum,
<verg. aen. 1.7>: Albanique patres, atque altae moenia Romae.
<verg. aen. 1.8>: Musa, mihi causas memora, quo numine laeso,


### doc_rows()

In [20]:
## Doc Rows - citation -> span mapping

catullus_docrows = next(T.doc_rows(catullus))

print('First 8 citation -> span mappings:')
for i, (citation, span) in enumerate(catullus_docrows.items()):
    if i >= 8:
        break
    print(f"  {citation}: {span.text[:40]}...")

First 8 citation -> span mappings:
  <cat. 1.1>: Cui dono lepidum nouum libellum...
  <cat. 1.2>: arido modo pumice expolitum?...
  <cat. 1.3>: Corneli, tibi; namque tu solebas...
  <cat. 1.4>: meas esse aliquid putare nugas,...
  <cat. 1.5>: iam tum cum ausus es unus Italorum...
  <cat. 1.6>: omne aeuum tribus explicare chartis,...
  <cat. 1.7>: doctis, Iuppiter, et laboriosis!...
  <cat. 1.8>: quare habe tibi quidquid hoc libelli...


### Metadata Filtering

In [21]:
# Solution: filter by exact author using metadata
# This gets ONLY Lucretius, not Anti-Lucretius

lucretius_files = [
    fileid for fileid, meta in T.metadata()
    if meta.get('author') == 'Lucretius'
]
pprint(lucretius_files)

['texts/lucretius.de_rerum_natura.part.1.tess',
 'texts/lucretius.de_rerum_natura.part.2.tess',
 'texts/lucretius.de_rerum_natura.part.3.tess',
 'texts/lucretius.de_rerum_natura.part.4.tess',
 'texts/lucretius.de_rerum_natura.part.5.tess',
 'texts/lucretius.de_rerum_natura.part.6.tess']


In [22]:
# Filter by genre (e.g., find all epic poetry)

epic_files = [
    fileid for fileid, meta in T.metadata()
    if meta.get('genre') == 'epic'
]
print(f"Epic texts: {len(epic_files)} files")
pprint(epic_files[:8])

Epic texts: 135 files
['texts/claudian.de_bello_gildonico.tess',
 'texts/claudian.de_bello_gothico.tess',
 'texts/claudian.de_consulatu_stilichonis.part.1.tess',
 'texts/claudian.de_consulatu_stilichonis.part.2.tess',
 'texts/claudian.de_consulatu_stilichonis.part.3.tess',
 'texts/claudian.de_raptu_proserpinae.part.1.tess',
 'texts/claudian.de_raptu_proserpinae.part.2.tess',
 'texts/claudian.de_raptu_proserpinae.part.3.tess']


In [23]:
# Filter by date: texts before 50 BCE (negative dates = BCE)

def get_date(meta):
    """Parse date string to int, handling missing/invalid values."""
    try:
        return int(meta.get('date', 0))
    except (ValueError, TypeError):
        return None

early_republic = [
    fileid for fileid, meta in T.metadata()
    if (d := get_date(meta)) is not None and d < -50
]
print(f"Texts before 50 BCE: {len(early_republic)} files")
pprint(early_republic[:8])

Texts before 50 BCE: 96 files
['texts/caesar.de_bello_gallico.part.1.tess',
 'texts/caesar.de_bello_gallico.part.2.tess',
 'texts/caesar.de_bello_gallico.part.3.tess',
 'texts/caesar.de_bello_gallico.part.4.tess',
 'texts/caesar.de_bello_gallico.part.5.tess',
 'texts/caesar.de_bello_gallico.part.6.tess',
 'texts/caesar.de_bello_gallico.part.7.tess',
 'texts/caesar.de_bello_gallico.part.8.tess']


In [24]:
# Top 5 genres in the corpus

from collections import Counter

genres = Counter(
    meta.get('genre') for _, meta in T.metadata()
    if meta.get('genre')
)
print("Top 5 genres:")
for genre, count in genres.most_common(5):
    print(f"  {genre}: {count} files")

Top 5 genres:
  historiography: 135 files
  epic: 135 files
  oratory: 84 files
  religious: 79 files
  miscellaneous: 70 files


In [25]:
# Combine filters: lyric poetry from the Augustan era

augustan_lyric = [
    fileid for fileid, meta in T.metadata()
    if meta.get('genre') == 'lyric'
    and (d := get_date(meta)) is not None 
    and -43 <= d <= 14
]
print(f"Augustan lyric poetry: {len(augustan_lyric)} files")
pprint(augustan_lyric[:8])

Augustan lyric poetry: 6 files
['texts/horace.carmen_saeculare.tess',
 'texts/horace.epodes.tess',
 'texts/horace.odes.part.1.tess',
 'texts/horace.odes.part.2.tess',
 'texts/horace.odes.part.3.tess',
 'texts/horace.odes.part.4.tess']


### search()

In [26]:
# search() - fast regex search across the corpus (no NLP required)
from itertools import islice

# Find lines mentioning Thebes (limit to first 5 results)
results = T.search(r'\bTheb\w+\b')
for fileid, citation, text, matches in islice(results, 5):
    print(f"{fileid} {citation}: found {matches}")
    print(f"  → {text[:60]}...")
    print()

texts/ammianus.rerum_gestarum.part.14.tess <amm. 14.11.15>: found ['Thebaeas']
  → Emensis itaque longis intervallis et planis, cum Hadrianopol...

texts/ammianus.rerum_gestarum.part.15.tess <amm. 15.10.9>: found ['Thebaeus']
  → Et primam Thebaeus Hercules, ad Geryonem exstinguendum (ut r...

texts/ammianus.rerum_gestarum.part.17.tess <amm. 17.4.2>: found ['Thebas', 'Thebais']
  → Urbem priscis saeculis conditam, ambitiosa moenium strue et ...

texts/ammianus.rerum_gestarum.part.19.tess <amm. 19.12.3>: found ['Thebaidis']
  → Materiam autem in infinitum quaestionibus extendendis dedit ...

texts/ammianus.rerum_gestarum.part.22.tess <amm. 22.16.1>: found ['Thebaida']
  → Tres provincias Aegyptus fertur habuisse temporibus priscis,...



### find_lines() / find_sents()

In [27]:
# find_lines() - find citation lines containing specific words/patterns

# Find lines with specific word forms
forms = ["Thebas", "Thebarum", "Thebis"]
for fileid, citation, text in islice(T.find_lines(forms=forms), 5):
    print(f"{citation}: {text[:70]}...")

<amm. 17.4.2>: Urbem priscis saeculis conditam, ambitiosa moenium strue et portarum c...
<amm. 22.16.2>: Igitur Thebais multas inter urbes clariores aliis Hermopolim habet, et...
<apul.fl. 22>: Crates ille Diogenis sectator, qui ut lar familiaris apud homines aeta...
<apul.met. 4.9>: Suscipit unus ex illo posteriore numero: Tune solus ignoras longe faci...


<aus. epit. 27.1>: THEBARUM regina fui, Sipyleia cautes...


In [28]:
# find_sents() - find sentences containing specific words
# Fast path: search by exact forms (uses regex, minimal NLP)

for hit in islice(T.find_sents(forms=["Caesar", "Caesarem", "Caesaris"]), 5):
    print(f"{hit['citation']}: {hit['sentence'][:80]}...")
    print(f"  Matched: {hit['matches']}")
    print()

<amm. 14.1.0>: Galli Caesaris saeuitia....
  Matched: ['Caesaris']

<amm. 14.1.1>: Post emensos insuperabilis expeditionis euentus, languentibus partium animis, qu...
  Matched: ['Caesaris']

<amm. 14.1.5>: sed quidquid Caesaris implacabilitati sedisset, id uelut fas iusque perpensum, c...
  Matched: ['Caesaris']

<amm. 14.1.6>: Hi peragranter et dissimulanter honoratorum circulis assistendo, peruadendoque d...
  Matched: ['Caesaris']

<amm. 14.1.10>: Quibus mox Caesar acrius efferatus, uelut contumaciae quoddam uexillum altius er...
  Matched: ['Caesar']



In [29]:
# find_sents() by lemma - slower but finds ALL forms
# Uses NLP to lemmatize, so it catches forms you might miss

# Find all sentences with any form of "bellum" (war)
for hit in islice(T.find_sents(lemma="bellum"), 5):
    print(f"{hit['citation']}: {hit['sentence'][:80]}...")
    print(f"  Matched forms: {hit['matches']}")
    print()

<amm. 14.2.1>: Namque et Isauri, quibus est usitatum saepe pacari, saepeque inopinis excursibus...
  Matched forms: ['bella']

<amm. 14.3.1>: Eo adducta re per Isauriam, rege Persarum bellis finitimis illigato, repellenteq...
  Matched forms: ['bellis']

<amm. 14.6.4>: Eius populus ab incunabulis primis ad usque pueritiae tempus extremum, quod anni...
  Matched forms: ['bella']

<amm. 14.6.4>: deinde aetatem ingressus adultam, post multiplices bellorum aerumnas, Alpes tran...
  Matched forms: ['bellorum']

<amm. 14.6.10>: Alii nullo quaerente, uultus seueritate assimulata, patrimonia sua in immensum e...
  Matched forms: ['bella']



In [30]:
# find_sents() with spaCy Matcher patterns - advanced pattern matching
# Search for ADJ + NOUN sequences (e.g., "magna voce", "pulchra puella")

pattern = [{"POS": "ADJ"}, {"POS": "NOUN"}]
for hit in islice(T.find_sents(matcher_pattern=pattern, fileids=T.fileids(match="catullus")), 5):
    print(f"{hit['citation']}: {hit['sentence'][:80]}...")
    print(f"  Matched: {hit['matches']}")
    print()

<cat. 1.1>: Cui dono lepidum nouum libellum arido modo pumice expolitum?...
  Matched: ['nouum libellum', 'arido modo']

<cat. 1.8>: quare habe tibi quidquid hoc libelli qualecumque, quod, o patrona uirgo, plus un...
  Matched: ['perenne saeclo']

<cat. 2.1>: Passer, deliciae meae puellae, quicum ludere, quem in sinu tenere, cui primum di...
  Matched: ['primum digitum', 'tristis animi']

<cat. 3.13>: at uobis male sit, malae tenebrae Orci, quae omnia bella deuoratis;...
  Matched: ['malae tenebrae']

<cat. 3.16>: o miselle passer!...
  Matched: ['miselle passer']



In [31]:
# More complex Matcher patterns
# Find sentences with a specific lemma followed by a noun

pattern = [{"LEMMA": "magnus"}, {"POS": "NOUN"}]
for hit in islice(T.find_sents(matcher_pattern=pattern), 5):
    print(f"{hit['citation']}: {hit['matches']}")

<amm. 14.2.8>: ['magna parte']
<amm. 14.2.13>: ['maiora uiribus']
<amm. 14.8.11>: ['magna protenta']


<amm. 15.3.7>: ['maiorum augurio']
<amm. 15.5.30>: ['magna industria']


### Morphological Agreement

In [32]:
# Build agreement-aware Matcher patterns using MORPH constraints
# Instead of post-hoc filtering, encode case/number/gender agreement
# directly in the pattern: both adj and noun must share the same features.

from spacy.matcher import Matcher
from itertools import product

caesar_files = T.fileids(match=r"caesar\.de_bello_gallico")

# Generate one pattern per case × number × gender combination (6×2×3 = 36)
cases = ["Nom", "Gen", "Dat", "Acc", "Abl", "Voc"]
numbers = ["Sing", "Plur"]
genders = ["Masc", "Fem", "Neut"]

patterns = [
    [
        {"LEMMA": "magnus", "MORPH": {"IS_SUPERSET": [f"Case={c}", f"Number={n}", f"Gender={g}"]}},
        {"POS": "NOUN",     "MORPH": {"IS_SUPERSET": [f"Case={c}", f"Number={n}", f"Gender={g}"]}},
    ]
    for c, n, g in product(cases, numbers, genders)
]

matcher = Matcher(T.nlp.vocab)
matcher.add("MAGNUS_NOUN_AGREE", patterns)

def get_citation(doc, idx):
    """Find the Tesserae citation for a token position."""
    for span in doc.spans.get("lines", []):
        if span.start <= idx < span.end:
            return span._.citation
    return doc._.fileid

# Find all agreement-verified magnus + NOUN pairs in Caesar
print("magnus + NOUN with morphological agreement (via Matcher MORPH constraints):\n")
count = 0
for doc in T.docs(caesar_files):
    for _, start, end in matcher(doc):
        adj, noun = doc[start], doc[start + 1]
        cit = get_citation(doc, start)
        print(f"{cit}:  {adj.text} {noun.text}")
        print(f"  {adj.text:>12}  {adj.morph}")
        print(f"  {noun.text:>12}  {noun.morph}")
        print()
        count += 1
        if count >= 5:
            break
    if count >= 5:
        break

print(f"(showing first 5 of {count}+ agreeing matches)")

magnus + NOUN with morphological agreement (via Matcher MORPH constraints):



<caes. gal. 1.2.5>:  magno dolore
         magno  Case=Abl|Gender=Masc|Number=Sing
        dolore  Case=Abl|Gender=Masc|Number=Sing

<caes. gal. 1.3.1>:  maximum numerum
       maximum  Case=Acc|Gender=Masc|Number=Sing
       numerum  Case=Acc|Gender=Masc|Number=Sing

<caes. gal. 1.4.2>:  magnum numerum
        magnum  Case=Acc|Gender=Masc|Number=Sing
       numerum  Case=Acc|Gender=Masc|Number=Sing

<caes. gal. 1.10.3>:  magnis itineribus
        magnis  Case=Abl|Gender=Neut|Number=Plur
    itineribus  Case=Abl|Gender=Neut|Number=Plur

<caes. gal. 1.12.3>:  magnam partem
        magnam  Case=Acc|Gender=Fem|Number=Sing
        partem  Case=Acc|Gender=Fem|Number=Sing

(showing first 5 of 5+ agreeing matches)


In [33]:
# Compare: the naive LEMMA+POS matcher (no MORPH constraints) catches
# non-agreeing pairs too. Post-hoc filtering with morph_agrees() separates them.
# Take care though... the results below are only as good as the tagger and morphological
# analyzer and so we should expect some false positives and false negatives in both 
# the agreeing and non-agreeing sets.

def morph_agrees(t1, t2, features=("Case", "Number", "Gender")):
    """Check if two tokens agree on morphological features.

    Returns True if all shared features match. Features missing from
    either token are treated as compatible (not penalized).
    """
    for feat in features:
        vals1, vals2 = set(t1.morph.get(feat)), set(t2.morph.get(feat))
        if vals1 and vals2 and not (vals1 & vals2):
            return False
    return True

naive_matcher = Matcher(T.nlp.vocab)
naive_matcher.add("MAGNUS_NOUN_NAIVE", [[{"LEMMA": "magnus"}, {"POS": "NOUN"}]])

agreeing, disagreeing = [], []
for doc in T.docs(caesar_files):
    for _, start, end in naive_matcher(doc):
        adj, noun = doc[start], doc[start + 1]
        cit = get_citation(doc, start)
        entry = (str(cit), adj.text, noun.text, str(adj.morph), str(noun.morph))
        if morph_agrees(adj, noun):
            agreeing.append(entry)
        else:
            disagreeing.append(entry)

print(f"Naive matcher (LEMMA+POS only, no MORPH constraints):")
print(f"  Total matches: {len(agreeing) + len(disagreeing)}")
print(f"  Agreeing: {len(agreeing)}, Non-agreeing: {len(disagreeing)}")

print(f"\nAgreeing pairs:")
for cit, a, n, am, nm in agreeing[:5]:
    print(f"  {cit:22}  {a} {n:15}  {am}")

if disagreeing:
    print(f"\nNon-agreeing pairs (false positives the MORPH matcher avoids):")
    for cit, a, n, am, nm in disagreeing[:5]:
        print(f"  {cit:22}  {a} {n}")
        print(f"  {'':22}    ADJ:  {am}")
        print(f"  {'':22}    NOUN: {nm}")

Naive matcher (LEMMA+POS only, no MORPH constraints):
  Total matches: 177
  Agreeing: 142, Non-agreeing: 35

Agreeing pairs:
  <caes. gal. 1.2.5>      magno dolore           Case=Abl|Gender=Masc|Number=Sing
  <caes. gal. 1.3.1>      maximum numerum          Case=Acc|Gender=Masc|Number=Sing
  <caes. gal. 1.4.2>      magnum numerum          Case=Acc|Gender=Masc|Number=Sing
  <caes. gal. 1.10.3>     magnis itineribus       Case=Abl|Gender=Neut|Number=Plur
  <caes. gal. 1.12.3>     magnam partem           Case=Acc|Gender=Fem|Number=Sing

Non-agreeing pairs (false positives the MORPH matcher avoids):
  <caes. gal. 1.43.4>     magnis hominum
                            ADJ:  Case=Abl|Gender=Fem|Number=Plur
                            NOUN: Case=Gen|Gender=Masc|Number=Plur
  <caes. gal. 1.46.4>     maius exercitui
                            ADJ:  Case=Nom|Gender=Neut|Number=Sing
                            NOUN: Case=Dat|Gender=Masc|Number=Sing
  <caes. gal. 1.50.1>     maioribus castris
  

### Noun Chunks

In [34]:
# Noun chunks require dependency parsing (FULL annotation level)
# Using Caesar's prose again for consistency

T_full = TesseraeReader(annotation_level=AnnotationLevel.FULL)

caesar_1 = T.fileids(match='caesar.de_bello_gallico.part.1.tess')[0]
doc = next(T_full.docs(caesar_1))

# Multi-word noun chunks with their morphological features
print(f"Multi-word noun chunks in {caesar_1}:\n")

count = 0
for chunk in doc.noun_chunks:
    if len(chunk) > 1:
        # Find citation
        cit = doc._.fileid
        for span in doc.spans.get("lines", []):
            if span.start <= chunk.start < span.end:
                cit = span._.citation
                break
        print(f"{cit}:  {chunk.text}")
        for token in chunk:
            print(f"  {token.text:>15}  {token.pos_:5}  {token.morph}")
        print()
        count += 1
        if count >= 5:
            break

Multi-word noun chunks in texts/caesar.de_bello_gallico.part.1.tess:

<caes. gal. 1.1.1>:  partes tres
           partes  NOUN   Case=Acc|Gender=Fem|Number=Plur
             tres  NUM    Case=Acc|Gender=Fem|Number=Plur

<caes. gal. 1.1.2>:  omnes lingua
            omnes  DET    Case=Nom|Gender=Fem|Number=Plur
           lingua  NOUN   Case=Abl|Gender=Fem|Number=Sing

<caes. gal. 1.1.2>:  Gallos ab Aquitanis
           Gallos  PROPN  Case=Acc|Gender=Masc|Number=Plur
               ab  ADP    
        Aquitanis  ADJ    Case=Abl|Gender=Masc|Number=Plur

<caes. gal. 1.1.2>:  Garumna flumen
          Garumna  PROPN  Case=Abl|Gender=Masc|Number=Sing
           flumen  NOUN   Case=Acc|Gender=Neut|Number=Sing

<caes. gal. 1.1.3>:  cultu atque humanitate prouinciae
            cultu  NOUN   Case=Abl|Gender=Masc|Number=Sing
            atque  CCONJ  
       humanitate  NOUN   Case=Abl|Gender=Fem|Number=Sing
       prouinciae  NOUN   Case=Gen|Gender=Fem|Number=Sing



In [35]:
# Noun chunks containing lemma MAGNUS: chunk starts with magnus, ends with a NOUN in agreement!
# This uses the dependency parse to find syntactically coherent phrases
# Skip 2-word chunks (already covered by the Matcher above) — show longer phrases
# Reuse morph_agrees() from above to verify first/last token agreement

print("Noun chunks with magnus … NOUN (agreeing) across all of Caesar's De Bello Gallico:\n")

caesar_files = T_full.fileids(match=r"caesar\.de_bello_gallico")

count = 0
for doc in T_full.docs(caesar_files):
    for chunk in doc.noun_chunks:
        if (len(chunk) > 2
            and chunk[0].lemma_ == "magnus"
            and chunk[-1].pos_ == "NOUN"
            and morph_agrees(chunk[0], chunk[-1])):
            # Find citation
            cit = doc._.fileid
            for span in doc.spans.get("lines", []):
                if span.start <= chunk.start < span.end:
                    cit = span._.citation
                    break
            print(f"{cit}:  {chunk.text}")
            for token in chunk:
                print(f"  {token.text:>15}  {token.pos_:6} {token.morph}")
            print()
            count += 1
            if count >= 5:
                break
    if count >= 5:
        break

print(f"Total: {count}+ noun chunks with magnus … NOUN in agreement (3+ tokens)")

Noun chunks with magnus … NOUN (agreeing) across all of Caesar's De Bello Gallico:

<caes. gal. 1.16.6>:  magna ex parte
            magna  ADJ    Case=Abl|Gender=Fem|Number=Sing
               ex  ADP    
            parte  NOUN   Case=Abl|Gender=Fem|Number=Sing

<caes. gal. 1.38.5>:  magnis nocturnis diurnisque itineribus
           magnis  ADJ    Case=Abl|Number=Plur
        nocturnis  ADJ    Case=Abl|Gender=Fem|Number=Plur
          diurnis  ADJ    Case=Abl|Gender=Fem|Number=Plur
              que  CCONJ  
       itineribus  NOUN   Case=Abl|Gender=Neut|Number=Plur

<caes. gal. 1.44.2>:  magnisque praemiis
           magnis  ADJ    Case=Abl|Number=Plur
              que  CCONJ  
         praemiis  NOUN   Case=Abl|Gender=Neut|Number=Plur



<caes. gal. 2.11.1>:  magno cum, strepitu
            magno  ADJ    Case=Abl|Gender=Masc|Number=Sing
              cum  ADP    
                ,  PUNCT  
         strepitu  NOUN   Case=Abl|Gender=Masc|Number=Sing

<caes. gal. 2.14.5>:  magnaeque uirtutis
           magnae  ADJ    Case=Gen|Gender=Fem|Number=Sing
              que  CCONJ  
         uirtutis  NOUN   Case=Gen|Gender=Fem|Number=Sing

Total: 5+ noun chunks with magnus … NOUN in agreement (3+ tokens)


### Concordance

In [36]:
## Concordance

# Build a concordance: word -> list of citations where it appears
catullus_conc = T.concordance(fileids=catullus, basis="lemma")

print(f"Unique lemmas in Catullus: {len(catullus_conc)}")

# Look up a specific lemma
if "amor" in catullus_conc:
    print("First 8 citations for 'amor':")
    for cit in catullus_conc["amor"][:8]:
        print(f"  {cit}")

Unique lemmas in Catullus: 3279
First 8 citations for 'amor':
  <cat. 6.16>
  <cat. 7.8>
  <cat. 10.1>
  <cat. 11.21>
  <cat. 13.9>
  <cat. 15.1>
  <cat. 21.4>
  <cat. 30.8>


In [37]:
# Concordance by surface text form (exact spelling)
catullus_conc_text = T.concordance(fileids=catullus, basis="text")

# Different forms of 'puella' (girl)
puella_forms = ["puella", "puellae", "puellam", "puellas", "puellis"]
print("Occurrences of 'puella' forms in Catullus:")
for form in puella_forms:
    if form in catullus_conc_text:
        count = len(catullus_conc_text[form])
        print(f"  {form}: {count} occurrences")

Occurrences of 'puella' forms in Catullus:
  puella: 19 occurrences
  puellae: 18 occurrences
  puellam: 2 occurrences
  puellis: 2 occurrences


### KWIC

In [38]:
# Basic KWIC search - find "amor" with 5 tokens of context on each side
for hit in T.kwic("amor", fileids=catullus, window=5, limit=5):
    print(f"{hit['left']} [{hit['match']}] {hit['right']}")
    print(f"  -- {hit['citation']}")
    print()

” hoc ut dixit , [Amor] , sinistra ut ante ,
  -- <cat. 45.8>

unquam contexit amores , nullus [amor] tali coniunxit foedere amantes qualis
  -- <cat. 64.335>

sub Latmia saxa relegans dulcis [amor] gyro deuocet aerio , idem
  -- <cat. 66.6>

semper concordia uestras , semper [amor] sedes incolat adsiduus . tu
  -- <cat. 66.88>

tuus in uita dulcis alebat [amor] . cuius ego interitu tota
  -- <cat. 68a.24>



In [39]:
# KWIC by lemma - finds all forms of a word (e.g., amo, amat, amant, amavit)
# Use by_lemma=True to match against lemmatized forms

for hit in T.kwic("amo", fileids=catullus, by_lemma=True, window=4, limit=5):
    print(f"{hit['left']} [{hit['match']}] {hit['right']}")
    print(f"  -- {hit['citation']}")
    print()

plus illa oculis suis [amabat] ; nam mellitus erat
  -- <cat. 3.5>

mea Lesbia , atque [amemus] , rumores que senum
  -- <cat. 5.1>

uentitabas quo puella ducebat [amata] nobis quantum amabitur nulla
  -- <cat. 8.5>

ducebat amata nobis quantum [amabitur] nulla . ibi illa
  -- <cat. 8.5>

bella ? quem nunc [amabis] ? cuius esse diceris
  -- <cat. 8.17>



### N-grams

In [40]:
# Extract bigrams (2-word sequences) from Catullus
from itertools import islice

bigrams = list(islice(T.ngrams(n=2, fileids=catullus), 8))
print("First 8 bigrams from Catullus:")
pprint(bigrams)

First 8 bigrams from Catullus:
['Cui dono',
 'dono lepidum',
 'lepidum nouum',
 'nouum libellum',
 'libellum arido',
 'arido modo',
 'modo pumice',
 'pumice expolitum']


In [41]:
# Most common bigrams
from collections import Counter

bigram_counts = Counter(T.ngrams(n=2, fileids=catullus))
for bigram, count in bigram_counts.most_common(8):
    print(f"  {bigram}: {count}")

  o Hymen: 26
  Hymen Hymenaee: 26
  o Hymenaee: 16
  ducentes subtegmina: 13
  tecum: 12
  currite ducentes: 12
  Hymenaee io: 11
  non est: 8


In [42]:
# Skipgrams - word pairs with 1-word gap
skipgrams = list(islice(T.skipgrams(n=2, k=1, fileids=catullus), 8))
pprint(skipgrams)

['Cui dono',
 'Cui lepidum',
 'dono lepidum',
 'dono nouum',
 'lepidum nouum',
 'lepidum libellum',
 'nouum libellum',
 'nouum arido']


### Export

In [43]:
# Export search results to TSV, CSV, or JSONL

results = T.find_sents(forms=["amor", "amoris", "amorem"], fileids=T.fileids(match="catullus"))
export = T.export_search_results(results, format="tsv")

print("TSV export (first 500 chars):")
print(export[:500])

TSV export (first 500 chars):
fileid	citation	matches	sentence
texts/catullus.carmina.tess	<cat. 11.21>	amorem	nec meum respectet, ut ante, amorem, qui illius culpa cecidit uelut prati ultimi flos, praetereunte postquam tactus aratro est.
texts/catullus.carmina.tess	<cat. 30.7>	amorem	certe tute iubebas animam tradere, inique, me inducens in amorem, quasi tuta omnia mi forent.
texts/catullus.carmina.tess	<cat. 45.8>	Amor	hoc ut dixit, Amor, sinistra ut ante, dextra sternuit adprobationem.
texts/catullus.carmina.tess	<cat. 55


### Annotation Levels

In [44]:
# AnnotationLevel controls how much NLP processing to apply

# NONE - use texts() for raw strings (fastest)
# TOKENIZE - tokenization + sentence boundaries only
# BASIC - adds lemmatization and POS tagging (default)
# FULL - full pipeline including NER and dependency parsing

# Create readers with different annotation levels
reader_fast = TesseraeReader(annotation_level=AnnotationLevel.TOKENIZE)
reader_full = TesseraeReader(annotation_level=AnnotationLevel.FULL)

print("Available annotation levels:")
for level in AnnotationLevel:
    print(f"  {level.name}: {level.value}")

Available annotation levels:
  NONE: none
  MINIMAL: minimal
  TOKENIZE: tokenize
  BASIC: basic
  FULL: full
